# 02d — Применение Random Forest к замещениям без строгих меток

Цель ноутбука — проверить, можно ли использовать supervised Random Forest для характеристики остальных замещений, и сопоставить его результат с текущим разделением Isolation Forest на `Expanded neutral-like` и `Out of neutral domain`.

> **Методологическая граница:** положительный вызов RF здесь означает *сходство с курированной группой Dataset 9 по используемым признакам*. Это не доказательство патогенности и не замена клинической классификации. Вызов Isolation Forest, в свою очередь, означает выход за изученный нейтральный домен, а не патогенность.

## Схема анализа

- класс 0: 8 280 чистых нейтральных вариантов Dataset 8;
- класс 1: 84 не перекрывающихся с Dataset 8 патогенных варианта Dataset 9;
- четыре варианта, одновременно попавшие в Dataset 8 и Dataset 9, исключены;
- разбиение выполнено по позиции mtDNA, поэтому одна позиция не попадает одновременно в обучение и тест;
- для каждого из 25 RF (5 повторов × 5 folds) порог калибруется отдельно на независимой нейтральной части обучения: 95-й процентиль RF score;
- остальные замещения скорируются всеми 25 моделями; основной вызов требует положительного решения не менее чем у 50% моделей, высокий консенсус — не менее чем у 80%;
- анализ повторён для девяти признаков и после удаления `phyloP100way`.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results' / 'model_random_forest_application'
FIGURE_DIR = ROOT / 'results' / 'figures' / 'model_random_forest_application'
RUN_APPLICATION = False  # True: полностью пересчитать ансамбль (~несколько минут)

if RUN_APPLICATION:
    subprocess.run(
        [sys.executable, str(ROOT / 'scripts' / 'random_forest_application.py')],
        cwd=ROOT,
        check=True,
    )

audit = pd.read_csv(RESULT_DIR / 'application_cohort_audit.tsv', sep='\t')
oof_metrics = pd.read_csv(RESULT_DIR / 'application_oof_metrics.tsv', sep='\t')
importance = pd.read_csv(RESULT_DIR / 'heldout_permutation_importance.tsv', sep='\t')
application_summary = pd.read_csv(RESULT_DIR / 'application_summary.tsv', sep='\t')
hybrid = pd.read_csv(RESULT_DIR / 'hybrid_if_rf_classification.tsv', sep='\t', low_memory=False)
crosstab = pd.read_csv(RESULT_DIR / 'hybrid_if_rf_crosstab.tsv', sep='\t')
profiles = pd.read_csv(RESULT_DIR / 'feature_profiles_by_rf_group.tsv', sep='\t')
top_candidates = pd.read_csv(RESULT_DIR / 'top_200_unlabeled_rf_candidates.tsv', sep='\t', low_memory=False)
decision = json.loads((RESULT_DIR / 'decision_summary.json').read_text(encoding='utf-8'))

print(f'Loaded {len(hybrid):,} application variants')

Loaded 41,336 application variants


## 1. Когорты и честность проверки

В RF не используются 41 280 вариантов без метки и 56 вариантов из disease-suspected post-hoc группы. Они остаются только application cohort. Это принципиально: Dataset 3 гетерогенен и не может целиком считаться положительным классом.

In [2]:
display(audit)

,rf_cohort,n_variants,n_positions
0,all_variants,49704,16568
1,clean_confirmed_pathogenic,84,80
2,clean_known_neutral,8280,4780
3,disease_suspected_posthoc,56,56
4,excluded_neutral_pathogenic_overlap,4,4
5,unlabeled,41280,14896


## 2. Качество RF на вариантах с известной строгой меткой

Это **out-of-fold**, а не качество на обучающих данных. В каждой итерации тестовая позиция полностью отсутствует в обучении. Из-за дисбаланса 84 против 8 280 основной показатель — average precision (AP), а не только ROC-AUC. Порог не равен 0,5 RF score: он каждый раз определяется по независимым нейтральным вариантам.

In [3]:
metric_order = ['roc_auc', 'average_precision', 'sensitivity', 'specificity', 'precision', 'f1', 'mcc', 'balanced_accuracy']
metric_table = (
    oof_metrics.pivot(index='metric', columns='feature_panel', values='value')
    .reindex(metric_order)
    .rename_axis(index='metric', columns=None)
)
display(metric_table.round(3))

,current_all_9,without_phyloP
metric,,
roc_auc,0.996,0.961
average_precision,0.797,0.285
sensitivity,0.988,0.798
specificity,0.954,0.952
precision,0.179,0.145
f1,0.303,0.245
mcc,0.411,0.327
balanced_accuracy,0.971,0.875


На всех девяти признаках RF действительно очень хорошо разделяет **две курированные когорты**: ROC-AUC 0,996 и AP 0,797. Однако precision при нейтральном FPR около 5% равна только 0,179 из-за соотношения классов примерно 99:1. Без phyloP ROC-AUC снижается до 0,961, AP — до 0,285, sensitivity — с 0,988 до 0,798. Следовательно, значительная часть apparent superiority RF зависит от phyloP.

## 3. Какой вклад делает каждый признак

У RF есть стандартная impurity importance, но она смещена в пользу признаков с большим числом возможных разбиений и измеряется на обучающей выборке. Поэтому здесь используется **held-out permutation importance**: на тестовых позициях один признак перемешивается, после чего измеряется падение AP. Чем больше падение, тем важнее признак для переноса на невидимые позиции. Отрицательное небольшое значение означает, что признак в среднем не помогал; это не биологический защитный эффект.

![Held-out permutation importance](../results/figures/model_random_forest_application/heldout_permutation_importance.png)

In [4]:
individual_ap = (
    importance.query("importance_type == 'individual' and metric == 'average_precision'")
    [['feature_panel', 'feature_or_family', 'mean_importance', 'sd_importance']]
    .sort_values(['feature_panel', 'mean_importance'], ascending=[True, False])
)
family_ap = (
    importance.query("importance_type == 'family' and metric == 'average_precision'")
    [['feature_panel', 'feature_or_family', 'mean_importance', 'sd_importance']]
    .sort_values(['feature_panel', 'mean_importance'], ascending=[True, False])
)
print('Отдельные признаки')
display(individual_ap.round({'mean_importance': 3, 'sd_importance': 3}))
print('Группы взаимосвязанных признаков')
display(family_ap.round({'mean_importance': 3, 'sd_importance': 3}))

Отдельные признаки
Группы взаимосвязанных признаков


,feature_panel,feature_or_family,mean_importance,sd_importance
22,current_all_9,phyloP100way,0.739,0.077
16,current_all_9,hom_rarity_soft,0.326,0.099
18,current_all_9,mlc_score,0.110,0.073
20,current_all_9,no_homoplasmic_signal,0.095,0.066
24,current_all_9,rarity_soft,0.040,0.049
12,current_all_9,codon_pos3_any,0.036,0.035
8,current_all_9,codon_pos1_any,0.017,0.029
14,current_all_9,het_rarity_soft,0.011,0.022
10,current_all_9,codon_pos2_any,0.006,0.019
42,without_phyloP,mlc_score,0.124,0.088


,feature_panel,feature_or_family,mean_importance,sd_importance
4,current_all_9,phylogenetic_conservation,0.723,0.095
6,current_all_9,population_rarity,0.676,0.116
2,current_all_9,mlc,0.094,0.084
0,current_all_9,codon_position,0.040,0.065
30,without_phyloP,population_rarity,0.213,0.085
28,without_phyloP,mlc,0.146,0.085
26,without_phyloP,codon_position,0.103,0.081


Главный признак полной RF — `phyloP100way`: его перестановка уменьшает AP в среднем на 0,739. Далее идёт `hom_rarity_soft` (0,327), затем `mlc_score` (0,110). После исключения phyloP на первое место выходят MLC и homoplasmic rarity, но их оценки заметно менее устойчивы.

Это важно не только статистически. Часть нейтральной выборки Dataset 8 была сформирована по низкому phyloP, а population-rarity признаки связаны с haplogroup/population критериями её формирования. RF способен восстановить правила курации обучающих классов, поэтому importance отвечает на вопрос «что разделяет наши Dataset 8 и 9», но не доказывает причинное влияние признака на патогенность.

## 4. Применение RF к остальным замещениям

Для каждого замещения сохранены средний score, интервал между 2,5-м и 97,5-м процентилями результатов 25 моделей, доля положительных RF-вызовов и consensus class. Интервал здесь отражает вариабельность между разбиениями/моделями, а не клинический доверительный интервал вероятности патогенности.

In [5]:
application_view = application_summary[[
    'feature_panel', 'rf_cohort', 'n_variants',
    'n_majority_pathogenic_like', 'majority_pathogenic_like_fraction',
    'n_high_consensus_pathogenic_like', 'high_consensus_pathogenic_like_fraction',
    'n_uncertain',
]].copy()
for column in ['majority_pathogenic_like_fraction', 'high_consensus_pathogenic_like_fraction']:
    application_view[column] = application_view[column].map(lambda value: f'{value:.1%}')
display(application_view)

,feature_panel,rf_cohort,n_variants,n_majority_pathogenic_like,majority_pathogenic_like_fraction,n_high_consensus_pathogenic_like,high_consensus_pathogenic_like_fraction,n_uncertain
0,current_all_9,disease_suspected_posthoc,56,32,57.1%,29,51.8%,7
1,current_all_9,unlabeled,41280,37083,89.8%,35766,86.6%,1938
2,without_phyloP,disease_suspected_posthoc,56,29,51.8%,18,32.1%,15
3,without_phyloP,unlabeled,41280,23611,57.2%,18618,45.1%,8868


![RF calls by Isolation Forest group](../results/figures/model_random_forest_application/rf_calls_by_isolation_group.png)

С полным набором признаков 37 083 из 41 280 вариантов без метки (89,8%) получают majority pathogenic-like call; 35 766 (86,6%) — высокий консенсус. Это слишком широкая группа для содержательной бинарной аннотации. Без phyloP положительных вызовов остаётся 23 611 (57,2%), высокий консенсус — у 18 618 (45,1%).

## 5. Чувствительность к phyloP

![RF panel agreement](../results/figures/model_random_forest_application/rf_panel_agreement.png)

In [6]:
agreement = (
    hybrid.query("rf_cohort == 'unlabeled'")['rf_panel_call_agreement']
    .value_counts()
    .rename_axis('agreement_class')
    .reset_index(name='n_variants')
)
agreement['fraction'] = agreement['n_variants'] / agreement['n_variants'].sum()
agreement['fraction'] = agreement['fraction'].map(lambda value: f'{value:.1%}')
display(agreement)

,agreement_class,n_variants,fraction
0,both_pathogenic_like,23505,56.9%
1,all9_only_pathogenic_like,13578,32.9%
2,neither_pathogenic_like,4091,9.9%
3,without_phyloP_only_pathogenic_like,106,0.3%


Только 23 505 вариантов положительны в обеих панелях. Для 13 578 вариантов положительный вызов существует лишь при наличии phyloP; обратных случаев всего 106. Поэтому полный RF нельзя интерпретировать как устойчивый новый классификатор всех замещений. Наиболее консервативная supervised-категория — согласованный положительный вызов обеих панелей, желательно с высоким консенсусом, но и она означает только приоритет для последующей проверки.

## 6. Совместное использование Isolation Forest и Random Forest

Две модели отвечают на разные вопросы:

- **Isolation Forest:** насколько вариант выходит за распределение нейтральной reference cohort;
- **Random Forest:** насколько вариант похож на курированные Dataset 9 pathogenic в сравнении с Dataset 8 neutral.

Поэтому естественный результат — две координаты, а не замена одной модели другой.

![Joint IF and RF scores](../results/figures/model_random_forest_application/isolation_vs_random_forest.png)

![Hybrid class counts](../results/figures/model_random_forest_application/hybrid_class_counts.png)

In [7]:
cross = (
    crosstab.groupby(['spectrum_group_primary_T95_preview', 'rf_panel_call_agreement'], as_index=False)
    ['n_variants'].sum()
    .pivot(index='spectrum_group_primary_T95_preview', columns='rf_panel_call_agreement', values='n_variants')
    .fillna(0).astype(int)
)
display(cross)

for group_name, group in hybrid.query("rf_cohort == 'unlabeled'").groupby('spectrum_group_primary_T95_preview'):
    positive_without_phyloP = group['without_phyloP__rf_majority_pathogenic_like'].mean()
    print(f'{group_name}: without-phyloP RF positive = {positive_without_phyloP:.1%} ({len(group):,} variants)')

expanded_neutral_like_T95: without-phyloP RF positive = 10.2% (13,631 variants)
unlabeled_out_of_neutral_domain_T95: without-phyloP RF positive = 80.4% (27,649 variants)


rf_panel_call_agreement,all9_only_pathogenic_like,both_pathogenic_like,neither_pathogenic_like,without_phyloP_only_pathogenic_like
spectrum_group_primary_T95_preview,,,,
expanded_neutral_like_T95,8194,1310,4050,77
unlabeled_out_of_neutral_domain_T95,5384,22195,41,29


Без phyloP RF уже гораздо лучше согласуется с геометрией нейтрального домена: он положителен примерно для 80,4% `Out of neutral domain`, но лишь для 10,2% `Expanded neutral-like`. Это хороший аргумент в пользу RF как **второй оси приоритизации**, но не независимая валидация — обе модели используют частично одинаковые признаки.

Полный девятипризнаковый RF даёт 27 579 `outside + RF-positive`, 9 504 `inside + RF-positive`, 4 127 `inside + RF-negative` и только 70 `outside + RF-negative`. Такая почти полная вложенность показывает, насколько phyloP усиливает положительные RF-вызовы.

## 7. Почему RF принимает такие решения

Ниже показаны медианы признаков в обучающих классах и среди вызовов полного RF. Это описание ассоциаций, не индивидуальное объяснение каждого варианта.

In [8]:
profile_cohorts = [
    'clean_known_neutral',
    'clean_confirmed_pathogenic',
    'unlabeled_rf_neutral_like',
    'unlabeled_rf_pathogenic_like',
]
profile_features = [
    'phyloP100way', 'mlc_score', 'hom_rarity_soft',
    'no_homoplasmic_signal', 'codon_pos3_any',
]
profile_table = (
    profiles[profiles['cohort'].isin(profile_cohorts) & profiles['feature'].isin(profile_features)]
    .pivot(index='feature', columns='cohort', values='median')
    .reindex(profile_features)
)
display(profile_table.round(3))

cohort,clean_confirmed_pathogenic,clean_known_neutral,unlabeled_rf_neutral_like,unlabeled_rf_pathogenic_like
feature,,,,
phyloP100way,5.146,-8.101,-2.853,2.632
mlc_score,0.535,0.000,0.000,0.549
hom_rarity_soft,6.000,4.048,4.435,6.000
no_homoplasmic_signal,1.000,0.000,0.000,1.000
codon_pos3_any,0.000,1.000,1.000,0.000


Например, median phyloP равна −8,10 в neutral, +5,15 в pathogenic, +2,63 в RF pathogenic-like unlabeled и −2,85 в RF neutral-like unlabeled. Аналогично RF-positive варианты имеют более высокий MLC, большую rarity и чаще не имеют homoplasmic signal. То есть направление решений биологически читаемо, но оно одновременно воспроизводит способ формирования исходных классов.

## 8. Итоговые выводы

1. **Random Forest лучше разделяет именно известные Dataset 8 и Dataset 9**, чем текущий one-class подход: OOF ROC-AUC 0,996 и AP 0,797 на девяти признаках. Это не результат на обучающих данных.
2. **Главный источник преимущества — phyloP.** Его held-out permutation importance намного выше остальных, а без него AP падает до 0,285. Поскольку низкий phyloP участвовал в формировании части neutral reference, здесь есть circularity относительно правил курации.
3. **Прямо заменить Isolation Forest на RF для классификации остальных замещений нельзя.** Полный RF называет положительными 89,8% unlabeled, а истинных меток для проверки этих предсказаний нет. RF score также нельзя читать как клиническую вероятность патогенности.
4. **Практически RF полезен как вторая ось.** Основное downstream-разделение `Expanded neutral-like / Out of neutral domain` следует сохранить по Isolation Forest, а RF-поля использовать для ранжирования внутри этих групп.
5. **Более строгий приоритет:** 17 993 варианта вне нейтрального домена, которые RF называет pathogenic-like в обеих feature panels с ≥80% межмодельным консенсусом. Это всё ещё широкая исследовательская категория, а не 17 993 доказанно патогенных варианта. Ещё 537 согласованных high-consensus вариантов находятся внутри IF-домена и представляют конфликт для отдельной проверки. Варианты `outside IF / RF-negative` могут представлять новизну, не похожую на известные pathogenic.
6. **Следующий методологический шаг** — внешняя валидация на независимом наборе патогенных и нейтральных вариантов, сформированном без phyloP/population criteria. Только после неё можно обсуждать RF как самостоятельный классификатор.

## 9. Таблица кандидатов для ручной проверки

Это первые строки из сохранённого ранжированного списка, а не автоматически подтверждённые pathogenic variants. Полная таблица: `results/model_random_forest_application/top_200_unlabeled_rf_candidates.tsv`.

In [9]:
candidate_columns = [
    'variant_id', 'position', 'spectrum_group_primary_T95_preview',
    'current_all_9__rf_call_fraction',
    'without_phyloP__rf_call_fraction',
    'rf_panel_call_agreement',
    'mlc_score', 'phyloP100way',
]
candidate_view = top_candidates[candidate_columns].head(20).copy()
candidate_view = candidate_view.round({
    'current_all_9__rf_call_fraction': 2,
    'without_phyloP__rf_call_fraction': 2,
    'mlc_score': 3,
    'phyloP100way': 2,
})
display(candidate_view)

,variant_id,position,spectrum_group_primary_T95_preview,current_all_9__rf_call_fraction,without_phyloP__rf_call_fraction,rf_panel_call_agreement,mlc_score,phyloP100way
0,m.6090T>A,6090,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.89
1,m.6090T>G,6090,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.89
2,m.9783T>A,9783,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.90
3,m.9783T>G,9783,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.90
4,m.11225G>A,11225,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.90
5,m.11225G>C,11225,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.90
6,m.11225G>T,11225,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.829,5.90
7,m.5690A>C,5690,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.830,5.89
8,m.5690A>T,5690,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.830,5.89
9,m.8099A>C,8099,unlabeled_out_of_neutral_domain_T95,1.0,1.0,both_pathogenic_like,0.831,5.90
